# 5-Fold Cross-Validation: Predicting Loss-Chasing Bet Increases

**Goal:** using the Bustabit gambling dataset, train a Random Forest to predict
whether a player increases their bet size right after losing a bet, and
evaluate it with 5-fold cross-validation.

Because the same players show up many times in this dataset, we use
GroupKFold (grouped by Username) instead of plain KFold, so a player's
bets never appear in both the training and validation sets at the same time.

*Note: "bet increased after a loss" is just a simple behavioral signal here,
not a clinical measure of gambling addiction.*


In [24]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_validate

from poker_coach.config import BUSTABIT_DATA_FILE


In [25]:
df = pd.read_csv(BUSTABIT_DATA_FILE, na_values=["NA"], parse_dates=["PlayDate"])
df["Won"] = df["CashedOut"].notna()

print(f"{len(df):,} bets from {df['Username'].nunique():,} players")
df.head()


50,000 bets from 4,149 players


,Id,GameID,Username,Bet,CashedOut,Bonus,Profit,BustedAt,PlayDate,Won
0,14196549,3366002,papai,5,1.20,0.0,1.00,8.24,2016-11-20 19:44:19+00:00,True
1,10676217,3343882,znay22,3,NaN,NaN,NaN,1.40,2016-11-14 14:21:50+00:00,False
2,15577107,3374646,rrrrrrrr,4,1.33,3.0,1.44,3.15,2016-11-23 06:39:15+00:00,True
3,25732127,3429241,sanya1206,10,NaN,NaN,NaN,1.63,2016-12-08 18:13:55+00:00,False
4,17995432,3389174,ADM,50,1.50,1.4,25.70,2.29,2016-11-27 08:14:48+00:00,True


In [26]:
df = df.sort_values(["Username", "PlayDate"]).reset_index(drop=True)
df.head()


,Id,GameID,Username,Bet,CashedOut,Bonus,Profit,BustedAt,PlayDate,Won
0,1764275,3304047,----------------,11,1.01,2.27,0.36,1.03,2016-11-03 06:14:27+00:00,True
1,2027336,3305330,----------------,12,1.01,1.83,0.34,3.25,2016-11-03 15:05:40+00:00,True
2,2194462,3306129,----------------,8,NaN,NaN,NaN,1.09,2016-11-03 20:26:24+00:00,False
3,911557,3299445,--dilib--,349,1.60,1.87,215.91,4.37,2016-11-01 22:57:26+00:00,True
4,961649,3299733,--dilib--,98,2.54,4.63,155.46,11.95,2016-11-02 00:56:16+00:00,True


In [27]:
df["PrevBet"] = df.groupby("Username")["Bet"].shift(1)
df["PrevWon"] = df.groupby("Username")["Won"].shift(1)
df["PrevBustedAt"] = df.groupby("Username")["BustedAt"].shift(1)

# drop first-ever bets (no previous-bet history available)
df = df.dropna(subset=["PrevBet", "PrevWon", "PrevBustedAt"]).reset_index(drop=True)

# keep only bets that came after a loss
loss_chase_df = df[df["PrevWon"] == False].copy()

# target: did the player bet more than last time?
loss_chase_df["BetIncreased"] = (loss_chase_df["Bet"] > loss_chase_df["PrevBet"]).astype(int)

print(f"{len(loss_chase_df):,} bets follow a loss, from "
      f"{loss_chase_df['Username'].nunique():,} players")
print(loss_chase_df["BetIncreased"].value_counts(normalize=True).round(3))


19,490 bets follow a loss, from 2,229 players
BetIncreased
0    0.598
1    0.402
Name: proportion, dtype: float64


In [28]:
feature_cols = ["PrevBet", "PrevBustedAt"]

X = loss_chase_df[feature_cols].reset_index(drop=True)
y = loss_chase_df["BetIncreased"].reset_index(drop=True)
groups = loss_chase_df["Username"].reset_index(drop=True)

print("X shape:", X.shape, "| y shape:", y.shape)


X shape: (19490, 2) | y shape: (19490,)


In [29]:
cv = GroupKFold(n_splits=5)
print(cv)


GroupKFold(n_splits=5, random_state=None, shuffle=False)


In [30]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model


,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

## 8. Run 5-Fold Cross-Validation

`cross_validate()` trains and evaluates the model once per fold (4 folds for
training, 1 for validation, rotating through all 5), and reports 5 metrics
for each fold:

- **accuracy** — overall percent of correct predictions
- **precision** — of the bets predicted "increased", how many really were
- **recall** — of the bets that really increased, how many did we catch
- **f1** — a balance between precision and recall
- **roc_auc** — how well the model ranks "increased" bets above "not
  increased" ones, across all thresholds

Passing `groups=groups` is what makes `cross_validate` actually use our
`GroupKFold` split correctly.


In [31]:
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

cv_results = cross_validate(model, X, y, cv=cv, groups=groups, scoring=scoring)


In [32]:
per_fold_df = pd.DataFrame({
    "fold": np.arange(1, 6),
    "accuracy": cv_results["test_accuracy"],
    "precision": cv_results["test_precision"],
    "recall": cv_results["test_recall"],
    "f1": cv_results["test_f1"],
    "roc_auc": cv_results["test_roc_auc"],
}).set_index("fold")

per_fold_df.round(4)


,accuracy,precision,recall,f1,roc_auc
fold,,,,,
1,0.5421,0.4242,0.3392,0.3770,0.5147
2,0.5467,0.4419,0.3477,0.3892,0.5217
3,0.5349,0.4074,0.3372,0.3690,0.5138
4,0.5439,0.3941,0.3407,0.3655,0.5162
5,0.5436,0.4065,0.3325,0.3658,0.5209


In [33]:
summary_df = per_fold_df.agg(["mean", "std"]).round(4)
summary_df


,accuracy,precision,recall,f1,roc_auc
mean,0.5422,0.4148,0.3394,0.3733,0.5175
std,0.0044,0.0185,0.0056,0.0100,0.0036


## 11. What the Results Mean

**Consistency across folds:** every metric's standard deviation is small
(0.004–0.019), so the model performs about the same no matter which players
land in the validation fold — the cross-validation estimate is stable.

**But the scores themselves are weak.** Accuracy averages 0.542, which sounds
okay until you notice that 59.8% of bets after a loss are *not* increased
(Step 4) — so simply guessing "not increased" every time would score ~59.8%,
beating this model. ROC-AUC (0.518) is the more honest metric here, and it's
barely above 0.5 (random guessing), meaning `PrevBet` and `PrevBustedAt` alone
carry very little signal about whether a given player will raise their next
bet after a loss. Recall (0.339) is the weakest number — the model misses
roughly two-thirds of the bets that actually did increase.

**Bottom line:** `GroupKFold` correctly tests the model on players it never
trained on, and it shows this particular two-feature baseline doesn't
generalize well to new players. That's still a useful result — it tells us
we'd need richer features (e.g. loss-streak length, recent betting trend) to
meaningfully predict this behavior, and it's a reminder that `BetIncreased` is
just a simple behavioral signal, not a diagnosis of gambling addiction.
